In [ ]:
import kagglehub

path = kagglehub.dataset_download("javiersanchezsoriano/roundabout-aerial-images-for-vehicle-detection")

print("Path to dataset files:", path)

In [ ]:
import shutil
import os

source_path = ''
destination_path = "/content/roundabout_dataset"

os.makedirs(destination_path, exist_ok=True)

if os.path.exists(destination_path):
    shutil.rmtree(destination_path)
shutil.copytree(source_path, destination_path)

print(f"Files copied to: {destination_path}")


In [ ]:
import os

print(os.listdir(destination_path))

In [ ]:
one_file="00031_frame000038_original.xml"
orignal="00030_frame000038_original.jpg"


In [ ]:
import xml.etree.ElementTree as ET
import cv2
import matplotlib.pyplot as plt
import os

def visualize_xml(image_path, xml_path):
    if not os.path.exists(image_path):
        print(f"Error: Image not found at {image_path}")
        return
    if not os.path.exists(xml_path):
        print(f"Error: XML not found at {xml_path}")
        return

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB for Matplotlib

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall('object'):
        class_name = obj.find('name').text
        bndbox = obj.find('bndbox')

        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)

        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)

        cv2.putText(image, class_name, (xmin, ymin - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

img_file = f"/content/roundabout_dataset/original/original/imgs/{orignal}"
xml_file = f"/content/roundabout_dataset/original/original/annotations/{one_file}"

visualize_xml(img_file, xml_file)

In [ ]:
import xml.etree.ElementTree as ET
import os

xml_sample_path = os.path.join(destination_path, 'original', 'original', 'annotations', one_file)

if not os.path.exists(xml_sample_path):
    print(f"Error: XML file not found at {xml_sample_path}")
else:
    tree = ET.parse(xml_sample_path)
    root = tree.getroot()

    print(f"Root element tag: {root.tag}")
    print("\nSample object annotations:")

    for obj in root.findall('object'):
        class_name = obj.find('name').text
        bndbox = obj.find('bndbox')
        xmin = bndbox.find('xmin').text
        ymin = bndbox.find('ymin').text
        xmax = bndbox.find('xmax').text
        ymax = bndbox.find('ymax').text
        print(f"  Class: {class_name}, Bounding Box: (xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax})")

In [ ]:
import xml.etree.ElementTree as ET
import os
import cv2

def convert_pascal_to_yolo(xml_path, image_path, class_mapping):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find('size')
    if size is not None:
        img_width = int(size.find('width').text)
        img_height = int(size.find('height').text)
    else:
        img = cv2.imread(image_path)
        if img is None:
            print(f"Warning: Could not read image {image_path}. Skipping.")
            return []
        img_height, img_width, _ = img.shape

    yolo_annotations = []

    for obj in root.findall('object'):
        class_name = obj.find('name').text
        bndbox = obj.find('bndbox')
        xmin = float(bndbox.find('xmin').text)
        ymin = float(bndbox.find('ymin').text)
        xmax = float(bndbox.find('xmax').text)
        ymax = float(bndbox.find('ymax').text)

        x_center = (xmin + xmax) / 2 / img_width
        y_center = (ymin + ymax) / 2 / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height

        class_id = class_mapping.get(class_name)
        if class_id is None:
            print(f"Warning: Class '{class_name}' not found in mapping. Skipping object.")
            continue

        yolo_annotations.append(f'{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')

    return yolo_annotations

yolo_labels_dir = os.path.join(destination_path, 'yolo_labels')
os.makedirs(yolo_labels_dir, exist_ok=True)

VOC_ROOT = os.path.join(destination_path, 'original', 'original')
annotations_dir = os.path.join(VOC_ROOT, 'annotations')
images_dir = os.path.join(VOC_ROOT, 'imgs')

class_map = {'vehicle': 0}

processed_count = 0
for xml_filename in os.listdir(annotations_dir):
    if xml_filename.endswith('.xml'):
        xml_filepath = os.path.join(annotations_dir, xml_filename)

        image_filename = xml_filename.replace('.xml', '.jpg')
        image_filepath = os.path.join(images_dir, image_filename)

        yolo_lines = convert_pascal_to_yolo(xml_filepath, image_filepath, class_map)

        yolo_label_filename = xml_filename.replace('.xml', '.txt')
        yolo_label_filepath = os.path.join(yolo_labels_dir, yolo_label_filename)

        if yolo_lines:
            with open(yolo_label_filepath, 'w') as f:
                for line in yolo_lines:
                    f.write(line + '\n')
            processed_count += 1

print(f"Successfully converted {processed_count} XML files to YOLO format in {yolo_labels_dir}")

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

yolo_dataset_root = os.path.join(destination_path, 'yolo_dataset')

images_train_dir = os.path.join(yolo_dataset_root, 'images', 'train')
labels_train_dir = os.path.join(yolo_dataset_root, 'labels', 'train')
images_val_dir = os.path.join(yolo_dataset_root, 'images', 'val')
labels_val_dir = os.path.join(yolo_dataset_root, 'labels', 'val')

os.makedirs(images_train_dir, exist_ok=True)
os.makedirs(labels_train_dir, exist_ok=True)
os.makedirs(images_val_dir, exist_ok=True)
os.makedirs(labels_val_dir, exist_ok=True)

print(f"YOLO dataset structure created at: {yolo_dataset_root}")

all_image_paths = [os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith('.jpg')]

all_label_paths = [os.path.join(yolo_labels_dir, f) for f in os.listdir(yolo_labels_dir) if f.endswith('.txt')]

aligned_files = []
for image_path in all_image_paths:
    image_filename = os.path.basename(image_path)
    label_filename = image_filename.replace('.jpg', '.txt')
    label_path = os.path.join(yolo_labels_dir, label_filename)

    if os.path.exists(label_path) and os.path.getsize(label_path) > 0: # Check if label file exists and is not empty
        aligned_files.append((image_path, label_path))
    else:
        pass

print(f"Found {len(aligned_files)} aligned image-label pairs.")

images, labels = zip(*aligned_files)


images_train, images_val, labels_train, labels_val = train_test_split(
    images, labels, test_size=0.2, random_state=42
)

print(f"Training images: {len(images_train)}")
print(f"Validation images: {len(images_val)}")

def move_files(file_list, destination_dir):
    for file_path in file_list:
        shutil.copy(file_path, destination_dir)

print("Moving training files...")
move_files(images_train, images_train_dir)
move_files(labels_train, labels_train_dir)
print(f"Moved {len(images_train)} training images and {len(labels_train)} training labels.")

print("Moving validation files...")
move_files(images_val, images_val_dir)
move_files(labels_val, labels_val_dir)
print(f"Moved {len(images_val)} validation images and {len(labels_val)} validation labels.")

print("Dataset organization complete!")

In [ ]:
try:
    import yaml
except ImportError:
    !pip install pyyaml
    import yaml

print("pyyaml library is ready.")

In [ ]:
import os
import yaml

data_yaml_path = os.path.join(yolo_dataset_root, 'data.yaml')

data_config = {
    'train': os.path.join('images', 'train'), # Corrected path to be relative
    'val': os.path.join('images', 'val'),     # Corrected path to be relative
    'nc': len(class_map), # Number of classes
    'names': list(class_map.keys()) # List of class names
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, indent=4)

print(f"data.yaml created successfully at: {data_yaml_path}")
print("Content of data.yaml:")
with open(data_yaml_path, 'r') as f:
    print(f.read())

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt') # 'n' stands for nano, a smaller and faster model

In [ ]:
import os
from ultralytics import YOLO

destination_path = "/content/roundabout_dataset"
yolo_dataset_root = os.path.join(destination_path, 'yolo_dataset')
data_yaml_path = os.path.join(yolo_dataset_root, 'data.yaml')

if not os.path.exists(data_yaml_path):
    print(f"Error: The data.yaml file was not found at '{data_yaml_path}'.")
    print("Please ensure that all previous data preparation steps have been successfully executed.")
    print("This includes downloading and copying the dataset (cells IeW_cZK-_hIa, qVATryLpMI8P),")
    print("converting annotations (cell e7d8e8ac), organizing the dataset (cell 4a06657e),")
    print("and generating the data.yaml file (cell 6233f112).")
else:
    model = YOLO('yolo11n.pt') # 'n' stands for nano, a smaller and faster model

    model.train(data=data_yaml_path,
                epochs=100, # Number of training epochs
                imgsz=640,  # Image size for training
                batch=16,   # Batch size
                name='yolov8_roundabout_detection') # Name for your training run results

In [ ]:
import os

yolo_runs_dir = 'runs/detect'

if os.path.exists(yolo_runs_dir):
    print(f"Contents of {yolo_runs_dir}:")
    for run_folder in os.listdir(yolo_runs_dir):
        run_path = os.path.join(yolo_runs_dir, run_folder)
        if os.path.isdir(run_path):
            print(f"  - {run_folder}/")
            weights_path = os.path.join(run_path, 'weights')
            if os.path.exists(weights_path):
                print(f"    Contents of {weights_path}:")
                for weight_file in os.listdir(weights_path):
                    print(f"      - {weight_file}")
            else:
                print("    (No 'weights' folder found in this run)")
else:
    print(f"The directory '{yolo_runs_dir}' does not exist. This might indicate that the training did not start or completed successfully, or the runtime environment was reset.")

In [ ]:
import os

print('Contents of current working directory:')
!ls -F

if os.path.exists('runs'):
    print('\nFound "runs" directory. Contents:')
    !ls -F runs
    if os.path.exists('runs/detect'):
        print('\nFound "runs/detect" directory. Contents:')
        !ls -F runs/detect
        if os.path.exists('runs/detect/yolov8_roundabout_detection'):
            print('\nFound "runs/detect/yolov8_roundabout_detection" directory. Contents:')
            !ls -F runs/detect/yolov8_roundabout_detection
            print('\nChecking weights folder:')
            !ls -F runs/detect/yolov8_roundabout_detection/weights/
        else:
            print('\n"yolov8_roundabout_detection" directory not found inside runs/detect.')
    else:
        print('\n"runs/detect" directory not found.')
else:
    print('\n"runs" directory not found. This indicates that the training either did not complete or the runtime was reset, deleting the output.')
    print('If "runs" is not found, you MUST re-execute the training cell (cell 1504ef90) to generate the training outputs and the last.pt file.')

In [ ]:
import os

yolo_dataset_root = "/content/roundabout_dataset/yolo_dataset"

if os.path.exists(yolo_dataset_root):
    print(f"Contents of {yolo_dataset_root}:")
    !ls -F {yolo_dataset_root}
else:
    print(f"The directory {yolo_dataset_root} does not exist. Please ensure previous steps to create the dataset structure have been executed.")